# Text classification: GPU vs Trainium

`AutoModelForSequenceClassification` (DistilRoBERTa emotion, 7 classes) on 1,024 messages.

Runs the **same script** on 1x NVIDIA A10G (EC2 Spot `g5.xlarge`) and on 1x AWS Trainium2 chip (EC2 Spot `trn2.3xlarge`), billed per second, and compares speed, results and cost. Run this notebook anywhere with AWS credentials (laptop, VS Code, SageMaker Studio, Colab).

In [ ]:
%pip install -q covex-neuron

## The only change from NVIDIA code

```diff
-device = "cuda" if torch.cuda.is_available() else "cpu"
+import covex_neuron as cn
+device = cn.device()
 model = AutoModelForSequenceClassification.from_pretrained(args.model).to(device).eval()
+model = cn.prepare(model, tokenizer)
```

`cn.device()` returns `"neuron"` on Trainium and `"cuda"` on a GPU. `cn.prepare()` applies bf16, fixed input shapes and `torch.compile(backend="neuron")` on Trainium and does nothing on a GPU, so this one file runs on both.

In [ ]:
import os
os.makedirs("job", exist_ok=True)

In [ ]:
%%writefile job/text_classify.py
"""Emotion classification with AutoModelForSequenceClassification, the NVIDIA script plus the covex lines."""

import argparse
import json
import os
import statistics
import time

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

parser = argparse.ArgumentParser()
parser.add_argument("--model", default="j-hartmann/emotion-english-distilroberta-base")
parser.add_argument("--n", type=int, default=1024)
parser.add_argument("--batch-size", type=int, default=32)
args = parser.parse_args()

MESSAGES = [
    "I can't believe they cancelled the show, I'm furious.",
    "We finally closed on the house today, best day ever!",
    "Walking alone at night down that street really scares me.",
    "My grandmother passed away last week and I miss her.",
    "Wow, I did not expect the package to arrive a week early.",
    "The meeting is at 3pm in the usual room.",
    "That smell from the fridge is absolutely disgusting.",
    "Thank you so much for helping me move, you're amazing.",
]
texts = [f"{MESSAGES[i % len(MESSAGES)]} #{i}" for i in range(args.n)]

import covex_neuron as cn  # covex: 1/2
device = cn.device()  # covex: "neuron" on Trainium

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(args.model)
model = AutoModelForSequenceClassification.from_pretrained(args.model).to(device).eval()
model = cn.prepare(model, tokenizer)  # covex: 2/2 bf16, fixed shapes, torch.compile(backend="neuron")
load_s = time.perf_counter() - t0


def classify(batch):
    enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return [model.config.id2label[i] for i in logits.argmax(-1).tolist()]


t0 = time.perf_counter()
classify(texts[: args.batch_size])
first_batch_s = time.perf_counter() - t0

classify(texts[:1])  # warm-up at batch size 1
latencies = []
for text in texts[:50]:
    t0 = time.perf_counter()
    classify([text])
    latencies.append((time.perf_counter() - t0) * 1000)

t0 = time.perf_counter()
labels = []
for i in range(0, len(texts), args.batch_size):
    labels += classify(texts[i : i + args.batch_size])
throughput = len(texts) / (time.perf_counter() - t0)

metrics = {
    "model": args.model, "device": device, "items": len(texts),
    "load_s": load_s, "first_batch_s": first_batch_s,
    "warm_latency_ms_p50": statistics.median(latencies),
    "warm_latency_ms_p95": statistics.quantiles(latencies, n=20)[-1],
    "throughput_items_per_s": throughput,
}
out = os.environ.get("SM_MODEL_DIR", "out/text_classify")
os.makedirs(out, exist_ok=True)
json.dump(metrics, open(f"{out}/metrics.json", "w"), indent=2)
json.dump(labels, open(f"{out}/predictions.json", "w"))
print(json.dumps(metrics, indent=2))

## Run on GPU and on Trainium

Each call starts a machine, runs the script, copies results to `covex-results/`, and shuts the machine down. Expect a few minutes each; the first Trainium run includes Neuron compilation.

In [ ]:
import covex_neuron as cn

gpu = cn.run(["job/text_classify.py"], on="gpu")

In [ ]:
trn = cn.run(["job/text_classify.py"], on="trainium")

## Compare

In [ ]:
cn.compare(gpu, trn)